# Module 1: Game Mechanics
### BWSI SGAI 2026

By the end of this module you'll know:
- What a **Humanoid** is, and what data it carries
- How the game **scores** your decisions (and which constants control that)
- Where **action costs** live and why the asymmetry between them is intentional
- How to change a single constant and immediately see the effect on the score

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## How do you play?

Each turn, a **Humanoid** appears. You choose one of four actions:

| Action | What happens | Time cost |
|--------|-------------|-----------|
| **Save** | Load them into the ambulance | 30 min |
| **Skip** | Leave them behind, move on | 15 min |
| **Squish** | Remove them from play (kills non-zombies) | 5 min |
| **Scram** | Drive to hospital, unload everyone, reset | 120 min |

The game ends when time runs out. Your goal: save as many people as possible without accidentally saving zombies.

## Six Ways to Play

The game has six modes — they differ in **who classifies** humanoids and **who picks actions**:

| Mode | Classifier | Recommender | Who acts |
|------|-----------|-------------|----------|
| `human` | none | none | human |
| `human_cnn` | CNN | none | human |
| `human_rl` | CNN (hidden) | RL | human |
| `human_cnn_rl` | CNN | RL | human |
| `machine` | CNN | RL | RL (human advances turns) |
| `auto` | CNN | RL | RL (headless, no window) |

`human_cnn_rl` is the richest interactive mode — you see both the CNN's guess and the RL's suggestion before deciding.  
`auto` is the one we'll use for experiments aimed at machine learning improvements — it runs fast, no window needed.

```
python main.py -m auto
```

## Where does this live in the code?

```
bwsi-sgai/
├── gameplay/           ← this module
│   ├── humanoid.py    ← now
│   ├── scorekeeper.py
│   ├── constants.py
│   └── enums.py
├── models/
├── game_interfaces/
├── ui/
├── training/
└── tests/
```

*`gameplay/` is the heart of the game rules. Nothing in here touches ML.*  
*Change things here and the effect shows up everywhere — including in training.*

*`bwsi-sgai/gameplay/humanoid.py`*

## What Is a Humanoid? — Locate

```
bwsi-sgai/
├── gameplay/
│   ├── humanoid.py    ← here
│   ├── scorekeeper.py
│   ├── constants.py
│   └── enums.py
```

Open `gameplay/humanoid.py` now. It's short — 88 lines.

**What it is:** a plain data class. Every person the ambulance encounters becomes one `Humanoid` object. You read attributes off it; you don't edit this file in normal use.

*`bwsi-sgai/gameplay/humanoid.py`*

## What Is a Humanoid? — Orient

```python
class Humanoid:
    fp: str      # path to the humanoid's image file
    state: str   # ground-truth state: "zombie", "healthy", "injured", "corpse"
    value: float # triage priority, 0.0–1.0 (assigned by DataParser)

    def __init__(self, fp: str, state: str, value: float = 0.0) -> None:
        self.fp = fp
        self.state = state
        self.value = value

    def is_zombie(self) -> bool: ...
    def is_injured(self) -> bool: ...
    def is_healthy(self) -> bool: ...
    def is_corpse(self) -> bool: ...

    # ... other methods ...
```

Three attributes. Four convenience methods. That's the whole interface.  
`state` comes from the `State` enum — we'll see that next.

*`bwsi-sgai/gameplay/humanoid.py`*

## What Is a Humanoid? — Touch

Run the cell below. You're creating a `Humanoid` by hand — the same way the game creates one when it reads the data CSV.

In [2]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [3]:
from gameplay.humanoid import Humanoid
from gameplay.enums import State

h = Humanoid(fp="data/zombie_002.jpg", state=State.ZOMBIE.value, value=0.3)

print(f"State:      {h.state}")
print(f"Value:      {h.value}")
print(f"Is zombie:  {h.is_zombie()}")
print(f"Is injured: {h.is_injured()}")
print(f"Is healthy: {h.is_healthy()}")

State:      zombie
Value:      0.3
Is zombie:  True
Is injured: False
Is healthy: False


> `value` (0–1) is a triage priority score. Higher means the game rewards saving this humanoid more.  
> The ML models **read** from `Humanoid` but never write to it — it's read-only data during a game.

## Now: state types and action costs

```
bwsi-sgai/
├── gameplay/
│   ├── humanoid.py
│   ├── scorekeeper.py
│   ├── constants.py
│   └── enums.py    ← now
├── models/
├── game_interfaces/
├── ui/
└── training/
```

*`enums.py` is where state types and action costs live. You'll need it if you ever add a new action.*

*`bwsi-sgai/gameplay/enums.py`*

## enums.py — Orient

```python
from enum import Enum

class State(Enum):        # the four humanoid types
    ZOMBIE  = "zombie"
    HEALTHY = "healthy"
    INJURED = "injured"
    CORPSE  = "corpse"

class ActionState(Enum):  # the four player actions
    SAVE   = "save"
    SQUISH = "squish"
    SKIP   = "skip"
    SCRAM  = "scram"

class ActionCost(Enum):   # time cost per action (minutes)
    SAVE   = 30
    SQUISH = 5
    SKIP   = 15
    SCRAM  = 120
```

**What's an enum?** A fixed set of named values — like a list that can't accidentally have typos.  
Instead of writing the string `"zombie"` everywhere (error-prone), code uses `State.ZOMBIE.value`.  
Python docs: [enum — Support for enumerations](https://docs.python.org/3/library/enum.html)

**Design intent in the docstring:**
> *SKIP (15 min) costs more than SQUISH (5 min) because skip represents navigating around the humanoid while leaving them behind — more time-consuming than a quick squish. This asymmetry matters for reward shaping: squishing is faster even when it kills, so the RL agent must learn to balance speed against humanitarian outcomes.*

That cost asymmetry is a deliberate design choice. Change it here and the RL agent's learned behavior will shift — without touching the model at all.

**One coupling rule to know:**
```python
# COUPLING: ActionState and ActionCost must stay in sync — same members, same order.
# Adding an action to one without the other produces a silent gap in cost lookups.
```
If you add a new action: add it to **both** `ActionState` and `ActionCost`.

## Now: constants — where experiments start

```
bwsi-sgai/
├── gameplay/
│   ├── humanoid.py
│   ├── scorekeeper.py
│   ├── constants.py    ← now
│   └── enums.py
├── models/
├── game_interfaces/
├── ui/
└── training/
```

*`constants.py` is the first file to open when you want to change how the game behaves.  
Every tunable parameter — game duration, ambulance size, reward weights — lives here.*

*`bwsi-sgai/gameplay/constants.py`*

## constants.py — Locate

Open `gameplay/constants.py`. It's 71 lines with comments. The structure:

```
# Game rules          → SHIFT_LENGTH, AMBULANCE_CAPACITY
# UI settings         → WINDOW_WIDTH, WINDOW_HEIGHT, LEGACY_CLOCK, UI_COLORS
# Paths               → DATA_DIR, CLASSIFIER_MODEL_PATH, RL_MODEL_PATH
# Reward weights      → HEADCOUNT_WEIGHT, VALUE_WEIGHT      ← we'll focus here
# Blocked action      → BLOCKED_ACTION_PENALTY
# CNN confidence      → CNN_CONFIDENCE_MID, CNN_CONFIDENCE_HIGH
# Observation space   → OBS_SIZE
```

Everything downstream imports from here. Change a value in `constants.py` and **every module that uses it picks up the change automatically** — you don't have to hunt through files.

*`bwsi-sgai/gameplay/constants.py`*

## constants.py — Orient

```python
# Game rules
SHIFT_LENGTH: int = 720           # total game time in minutes
AMBULANCE_CAPACITY: int = 10      # max humanoids the ambulance can hold at once

# Reward weights — must sum to 1. Adjust to shift agent priorities:
#   HEADCOUNT_WEIGHT=1.0 → pure headcount (saved − killed), ignores value
#   HEADCOUNT_WEIGHT=0.0 → pure value signal, ignores headcount
HEADCOUNT_WEIGHT: float = 0.5     # weight on (saved − killed)
VALUE_WEIGHT: float = 1.0 - HEADCOUNT_WEIGHT   # weight on saved_value

# Penalty applied when the agent attempts a blocked action (e.g. save when full).
BLOCKED_ACTION_PENALTY: float = -0.5

# ... other constants ...
```

Notice: `VALUE_WEIGHT` is derived from `HEADCOUNT_WEIGHT` — **you only change one number** and both stay in sync.  
The comment tells you the valid range and what each extreme means.

*`bwsi-sgai/gameplay/constants.py`*

## constants.py — Touch

Run the cell below to read the current values directly from the file.

In [ ]:
from gameplay.constants import (
    SHIFT_LENGTH,
    AMBULANCE_CAPACITY,
    HEADCOUNT_WEIGHT,
    VALUE_WEIGHT,
    BLOCKED_ACTION_PENALTY,
)

print(f"SHIFT_LENGTH         = {SHIFT_LENGTH} min  ({SHIFT_LENGTH / 60:.1f} hours)")
print(f"AMBULANCE_CAPACITY   = {AMBULANCE_CAPACITY} slots")
print(f"HEADCOUNT_WEIGHT     = {HEADCOUNT_WEIGHT}")
print(f"VALUE_WEIGHT         = {VALUE_WEIGHT}  (= 1.0 - HEADCOUNT_WEIGHT)")
print(f"BLOCKED_ACTION_PENALTY = {BLOCKED_ACTION_PENALTY}")

## Now: how the game keeps score

```
bwsi-sgai/
├── gameplay/
│   ├── humanoid.py
│   ├── scorekeeper.py  ← now
│   ├── constants.py    ← now
│   └── enums.py
├── models/
├── game_interfaces/
├── ui/
└── training/
```

*`ScoreKeeper` owns time, score, and the ambulance manifest. `constants.py` sets the weights.*

*`bwsi-sgai/gameplay/scorekeeper.py`*

## ScoreKeeper — Locate

Open `gameplay/scorekeeper.py`. The class owns:

```
ScoreKeeper
├── State
│   ├── remaining_time    — minutes left in shift
│   ├── ambulance         — {'zombie': 0, 'injured': 0, 'healthy': 0, 'corpse': 0}
│   ├── score             — {'killed': 0, 'saved': 0, 'saved_value': 0.0}
│   └── capacity          — max ambulance slots
├── Actions
│   ├── save(humanoid)    — load onto ambulance, deduct 30 min
│   ├── squish(humanoid)  — destroy, deduct 5 min
│   ├── skip(humanoid)    — leave behind, deduct 15 min
│   └── scram(humanoid)   — drive to hospital, deduct 120 min
└── Scoring
    ├── get_score()              — final kill/save/saved_value totals
    └── get_cumulative_reward()  — weighted reward (used by RL training)
```

*`bwsi-sgai/gameplay/scorekeeper.py`*

## ScoreKeeper — Orient: the reward formula

```python
def get_cumulative_reward(self) -> float:
    killed     = self.score["killed"]
    saved      = self.score["saved"]
    saved_value = self.score["saved_value"]

    if self.ambulance["zombie"] > 0:
        killed += self.ambulance["injured"] + self.ambulance["healthy"]
        # ambulance_value is lost — zombie contamination wipes all value
    else:
        saved += self.ambulance["injured"] + self.ambulance["healthy"]
        saved_value += self.ambulance_value

    # Reward = HEADCOUNT_WEIGHT × (saved − killed) + VALUE_WEIGHT × saved_value
    return HEADCOUNT_WEIGHT * (saved - killed) + VALUE_WEIGHT * saved_value
```

**The formula:**
```
reward = HEADCOUNT_WEIGHT × (saved − killed) + VALUE_WEIGHT × saved_value
```

Both weights are imported directly from `gameplay/constants.py` — change them there, and the reward changes everywhere (UI, training, evaluation).

**Why two separate weights?**

Two different signals are being traded off:
- `saved − killed` counts *people* — every life is worth the same
- `saved_value` counts *priority* — saving a high-value person is worth more

With `HEADCOUNT_WEIGHT = 1.0`: the agent learns to maximize headcount, ignoring priority. It might save 5 low-value healthy people instead of 1 high-value injured person.  
With `HEADCOUNT_WEIGHT = 0.0`: the agent ignores headcount entirely and chases value — it might skip 10 low-value people to wait for a high-value one.

The default `0.5 / 0.5` split means both signals matter equally. **This is the design decision you can actually test.**

*`bwsi-sgai/gameplay/scorekeeper.py`*

## ScoreKeeper — Touch

Run the cell below. You're creating a `ScoreKeeper` in isolation — no game loop needed — and checking its starting state.

In [4]:
from gameplay.scorekeeper import ScoreKeeper
from gameplay.constants import SHIFT_LENGTH, AMBULANCE_CAPACITY

sk = ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY, seed=42)

print("Starting state:")
print(f"  get_score()    = {sk.get_score()}")
print(f"  remaining_time = {sk.remaining_time} min")
print(f"  capacity       = {sk.capacity} slots")
print(f"  ambulance      = {sk.ambulance}")

Starting state:
  get_score()    = {'killed': 0, 'saved': 0, 'saved_value': 0.0}
  remaining_time = 720 min
  capacity       = 10 slots
  ambulance      = {'zombie': 0, 'injured': 0, 'healthy': 0, 'corpse': 0}


INSTRUCTOR NOTES:
Point out that seed=42 here records the seed in summary.csv for audit purposes only.
It does NOT control which humanoids appear or in what order — that's DataParser's seed.
Ask: what would you change to give the agent more or less time per shift?

*`bwsi-sgai/gameplay/scorekeeper.py`*

## ScoreKeeper — Formula in action

Run the cell below. Watch the reward formula react to a zombie contamination event.

In [5]:
from gameplay.scorekeeper import ScoreKeeper
from gameplay.constants import SHIFT_LENGTH, AMBULANCE_CAPACITY, HEADCOUNT_WEIGHT, VALUE_WEIGHT
from gameplay.humanoid import Humanoid
from gameplay.enums import State

sk = ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY, seed=42)

# Save 3 injured humanoids
for i in range(3):
    sk.save(Humanoid(fp=f"data/injured_{i:03d}.jpg", state=State.INJURED.value, value=0.8))

reward_before = sk.get_cumulative_reward()
print(f"After saving 3 injured → reward = {reward_before:.2f}")
print(f"  ambulance = {sk.ambulance}")

# Now accidentally save a zombie
sk2 = ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY, seed=42)
for i in range(3):
    sk2.save(Humanoid(fp=f"data/injured_{i:03d}.jpg", state=State.INJURED.value, value=0.8))
sk2.save(Humanoid(fp="data/zombie_001.jpg", state=State.ZOMBIE.value, value=0.0))

reward_after = sk2.get_cumulative_reward()
print(f"\nAfter adding 1 zombie → reward = {reward_after:.2f}")
print(f"  ambulance = {sk2.ambulance}")
print(f"\nReward swing from zombie contamination: {reward_after - reward_before:.2f}")
print(f"\n(Using HEADCOUNT_WEIGHT={HEADCOUNT_WEIGHT}, VALUE_WEIGHT={VALUE_WEIGHT})")

After saving 3 injured → reward = 2.70
  ambulance = {'zombie': 0, 'injured': 3, 'healthy': 0, 'corpse': 0}

After adding 1 zombie → reward = -1.50
  ambulance = {'zombie': 1, 'injured': 3, 'healthy': 0, 'corpse': 0}

Reward swing from zombie contamination: -4.20

(Using HEADCOUNT_WEIGHT=0.5, VALUE_WEIGHT=0.5)


## Exercise: Change This and Run

**Time: under 5 minutes**

The scoring formula uses two weights defined in `gameplay/constants.py`:

```
reward = HEADCOUNT_WEIGHT × (saved − killed) + VALUE_WEIGHT × saved_value
```

Both default to `0.5`. **Your task:**

1. Open `gameplay/constants.py`
2. Change `HEADCOUNT_WEIGHT` to `1.0`
3. Save the file
4. **Restart the kernel** (Kernel → Restart Kernel)
5. Run **all cells** in this section (the two below)
6. Compare the reward values — what changed?

Then try `HEADCOUNT_WEIGHT = 0.0` and repeat.

In [6]:
# Run this cell first — imports use the current values from constants.py
from gameplay.scorekeeper import ScoreKeeper
from gameplay.constants import SHIFT_LENGTH, AMBULANCE_CAPACITY, HEADCOUNT_WEIGHT, VALUE_WEIGHT
from gameplay.humanoid import Humanoid
from gameplay.enums import State

print(f"Loaded: HEADCOUNT_WEIGHT={HEADCOUNT_WEIGHT}, VALUE_WEIGHT={VALUE_WEIGHT}")

Loaded: HEADCOUNT_WEIGHT=0.5, VALUE_WEIGHT=0.5


In [7]:
# Simulated outcome: saved 4 humanoids, killed 1, with varying values
# Change HEADCOUNT_WEIGHT in constants.py, restart the kernel, run both cells again

sk = ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY, seed=42)

# Simulate: 4 saves (mix of injured+healthy), 1 squish of healthy (= 1 kill)
humanoids_to_save = [
    Humanoid("data/injured_001.jpg", State.INJURED.value, value=0.9),
    Humanoid("data/healthy_001.jpg", State.HEALTHY.value, value=0.4),
    Humanoid("data/injured_002.jpg", State.INJURED.value, value=0.7),
    Humanoid("data/healthy_002.jpg", State.HEALTHY.value, value=0.2),
]
for h in humanoids_to_save:
    sk.save(h)

# One squish of a healthy (counts as a kill)
sk.squish(Humanoid("data/healthy_003.jpg", State.HEALTHY.value, value=0.6))

# Scram to finalize
sk.scram()
final = sk.score

print("=== Final outcome ===")
print(f"  killed       = {final['killed']}")
print(f"  saved        = {final['saved']}")
print(f"  saved_value  = {final['saved_value']:.2f}")

reward = HEADCOUNT_WEIGHT * (final['saved'] - final['killed']) + VALUE_WEIGHT * final['saved_value']
print(f"\n  reward = {HEADCOUNT_WEIGHT} × ({final['saved']} − {final['killed']}) + {VALUE_WEIGHT} × {final['saved_value']:.2f}")
print(f"         = {reward:.4f}")
print(f"\n>>> Try changing HEADCOUNT_WEIGHT in constants.py and re-running <<<")

=== Final outcome ===
  killed       = 1
  saved        = 4
  saved_value  = 2.20

  reward = 0.5 × (4 − 1) + 0.5 × 2.20
         = 2.6000

>>> Try changing HEADCOUNT_WEIGHT in constants.py and re-running <<<


## Bonus: Run a Full Auto Game

After changing `HEADCOUNT_WEIGHT` in `constants.py`, run a real game in headless mode from the terminal:

```bash
python main.py -m auto
```

This uses the saved RL weights with your new reward weights.  
The game writes results to `summary.csv` — open it to see the `reward` column.  

Run twice (once with `0.5`, once with `1.0`) and compare the `reward` and `saved_value` columns side by side.

INSTRUCTOR NOTES:
Walk through the reward formula on the whiteboard before students run the exercise.
Ask: if HEADCOUNT_WEIGHT = 0.0 and you save 10 very low-value injured people vs 2 high-value injured people, which scores better?
The RL agent was trained with HEADCOUNT_WEIGHT=0.5 — changing it here changes evaluation but not the agent's internal policy.
Students who finish early: try HEADCOUNT_WEIGHT=0.0, HEADCOUNT_WEIGHT=1.0, and see if the ranking of a saved vs killed game flips.

## Pop Quiz — Navigation

These are navigation questions, not recall. Which file do you open?

---

**Q1.** You want to change how much time it costs to squish a humanoid.

A) `gameplay/constants.py`  
B) `gameplay/enums.py`  
C) `gameplay/scorekeeper.py`  
D) `gameplay/humanoid.py`

**Answer: B — `gameplay/enums.py`**  
`ActionCost.SQUISH = 5`. That's where all action time costs live.  
`constants.py` holds reward weights and game configuration, not action timing.

## Pop Quiz — Navigation

**Q2.** You want to make the ambulance hold 20 humanoids instead of 10.

A) `gameplay/enums.py`  
B) `gameplay/humanoid.py`  
C) `gameplay/constants.py`  
D) `gameplay/scorekeeper.py`

**Answer: C — `gameplay/constants.py`**  
`AMBULANCE_CAPACITY: int = 10`. Change it there and `ScoreKeeper` picks it up automatically — no other edits needed.

## Pop Quiz — Navigation

**Q3.** You want to add a fifth humanoid state called `"mutant"` to the game.

A) `gameplay/constants.py`  
B) `gameplay/humanoid.py`  
C) `gameplay/scorekeeper.py`  
D) `gameplay/enums.py`

**Answer: D — `gameplay/enums.py`**  
`State` is defined there. You'd add `MUTANT = "mutant"` to the enum.  
But that's only step one — you'd also need to add a `is_mutant()` method to `Humanoid` and handle the new state in `ScoreKeeper`'s action methods. Enums are the entry point, not the whole change.

## What You Now Know

- `gameplay/humanoid.py` — data class; **read from it**, don't edit it during normal use
- `gameplay/enums.py` — state types and action costs; **edit here to add an action**
- `gameplay/constants.py` — all tunable parameters; **start here for any experiment**
- `gameplay/scorekeeper.py` — owns time, score, ambulance; the reward formula lives in `get_cumulative_reward()`

The reward formula:
```
reward = HEADCOUNT_WEIGHT × (saved − killed) + VALUE_WEIGHT × saved_value
```
is controlled entirely by one number in `constants.py`.

---

**Next: Module 2 — Data**  
Where do the humanoid images and labels come from? How does the game decide who shows up?  
We'll open `data/consolidated_metadata.csv` and trace how `DataParser` turns rows into `Humanoid` objects.